# 20 Markowitz 均值方差优化：构建有效前沿

> **模块 2.3 — 风控与优化** | 理论学习 2h · 实战 3h

本 Notebook 从零实现 Markowitz 均值方差模型，用三种方法（解析解、数值优化、PyPortfolioOpt）构建有效前沿，找到最优组合权重，并深入理解分散化的数学原理。

## 🎯 学习目标

1. **理解 Markowitz 框架**：期望收益 vs 风险（方差）的权衡，有效前沿的几何意义
2. **拉格朗日解析解**：手推最小方差组合的闭式解，理解矩阵形式的优雅
3. **数值优化**：用 `scipy.optimize` 求解带约束的组合优化问题
4. **PyPortfolioOpt 实战**：用专业库构建有效前沿，快速找到最优组合
5. **约束条件分析**：做空限制、权重上下限、行业约束对有效前沿的影响

## 📦 环境依赖

```python
numpy, pandas, matplotlib, scipy, yfinance, PyPortfolioOpt
```

> ⚠️ 数据获取包含降级：yfinance 不可用时自动使用模拟数据。PyPortfolioOpt 不可用时仅用 scipy 实现。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print("✓ 环境就绪")

## 1. Markowitz 均值方差理论

### 核心思想

Harry Markowitz（1952）提出的框架回答：**给定一组资产，如何分配权重以在给定风险下最大化收益？**

### 数学表述

投资组合由权重向量 $\mathbf{w} = (w_1, ..., w_n)^T$ 描述，满足 $\sum w_i = 1$。

- **组合期望收益**：$\mu_p = \mathbf{w}^T \boldsymbol{\mu}$
- **组合方差**：$\sigma_p^2 = \mathbf{w}^T \boldsymbol{\Sigma} \mathbf{w}$
- **夏普比率**：$S_p = \frac{\mu_p - r_f}{\sigma_p}$

### 有效前沿（Efficient Frontier）

在风险-收益平面上，**给定期望收益下风险最小的组合**构成的曲线。

### 两个关键组合

| 组合 | 目标 | 含义 |
|------|------|------|
| **最小方差组合（MVP）** | $\min \mathbf{w}^T\Sigma\mathbf{w}$ | 不考虑收益，纯最小化风险 |
| **最大夏普比组合（MSR）** | $\max (\mu_p - r_f)/\sigma_p$ | 风险调整后收益最高 |
| **切线组合（Tangency）** | 过无风险利率与有效前沿的切线 | 同 MSR |

## 2. 数据准备

选择 5 只 A 股构建投资组合，获取近 2 年日收益率数据。

In [ ]:
# ========== 数据获取（带降级） ==========
def get_stock_data():
    try:
        import yfinance as yf
        tickers = ['600519.SS', '000858.SZ', '601318.SS', '600036.SS', '300750.SZ']
        names = ['贵州茅台', '五粮液', '中国平安', '招商银行', '宁德时代']
        data = pd.DataFrame()
        for t in tickers:
            df = yf.download(t, start='2022-01-01', progress=False)
            if df.empty: raise Exception(f"{t} 数据为空")
            data[t] = df['Adj Close']
        returns = data.pct_change().dropna()
        print("✓ 使用真实数据 (yfinance)")
        return returns, names
    except Exception as e:
        print(f"yfinance 不可用: {e}")
        print("→ 降级为模拟数据...")
        np.random.seed(42)
        n_days, n_assets = 500, 5
        names = ['模拟资产A','模拟资产B','模拟资产C','模拟资产D','模拟资产E']
        returns = pd.DataFrame(np.random.standard_t(df=4, size=(n_days, n_assets))*0.015,
                               columns=[f'Asset_{i}' for i in range(n_assets)])
        corr = np.array([[1.0,0.6,0.5,0.4,0.3],[0.6,1.0,0.4,0.3,0.3],[0.5,0.4,1.0,0.5,0.2],[0.4,0.3,0.5,1.0,0.2],[0.3,0.3,0.2,0.2,1.0]])
        L = np.linalg.cholesky(corr)
        returns = returns @ L.T
        returns.columns = [f'Asset_{i}' for i in range(n_assets)]
        print("✓ 使用模拟数据 (t分布 + Cholesky相关性)")
        return returns, names

returns, stock_names = get_stock_data()
returns.index = pd.date_range(end='2024-12-31', periods=len(returns), freq='B')

# 年化参数
mu_annual = returns.mean() * 252
sigma_annual = returns.std() * np.sqrt(252)
cov_annual = returns.cov() * 252

print(f"\n数据概览：{len(returns)}个交易日, {returns.shape[1]}只股票")
print("\n年化收益率与波动率：")
summary = pd.DataFrame({'年化收益': mu_annual, '年化波动': sigma_annual, '夏普比(无风险=2%)': (mu_annual-0.02)/sigma_annual})
display(summary.round(4))

# 相关性热力图
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(returns.corr(), cmap='RdYlGn', vmin=-1, vmax=1)
for i in range(len(stock_names)):
    for j in range(len(stock_names)):
        ax.text(j, i, f'{returns.corr().iloc[i,j]:.2f}', ha='center', va='center', fontsize=9)
ax.set_xticks(range(len(stock_names))); ax.set_yticks(range(len(stock_names)))
ax.set_xticklabels(stock_names, rotation=45, ha='right')
ax.set_yticklabels(stock_names)
ax.set_title('资产相关性矩阵')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight'); plt.show()

## 3. 方法一：拉格朗日解析解（最小方差组合）

### 数学推导

最小方差组合问题：
$$\min_{\mathbf{w}} \frac{1}{2}\mathbf{w}^T\Sigma\mathbf{w} \quad \text{s.t.} \quad \mathbf{1}^T\mathbf{w} = 1$$

拉格朗日函数：
$$\mathcal{L} = \frac{1}{2}\mathbf{w}^T\Sigma\mathbf{w} + \lambda(1 - \mathbf{1}^T\mathbf{w})$$

一阶条件 → **闭式解**：
$$\mathbf{w}_{MVP} = \frac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}^T\Sigma^{-1}\mathbf{1}}$$

> 这就是"1/n 的优化版"——不是简单等权，而是根据协方差结构分配权重。

In [ ]:
# ========== 解析解：最小方差组合 ==========
Sigma = returns.cov().values * 252  # 年化协方差
ones = np.ones(len(stock_names))
Sigma_inv = np.linalg.inv(Sigma)

# 闭式解
w_mvp_analytic = Sigma_inv @ ones / (ones.T @ Sigma_inv @ ones)

# 计算组合指标
mu_mvp = (mu_annual.values @ w_mvp_analytic)
sigma_mvp = np.sqrt(w_mvp_analytic.T @ Sigma @ w_mvp_analytic)
sharpe_mvp = (mu_mvp - 0.02) / sigma_mvp

print("="*60)
print("📊 最小方差组合（拉格朗日解析解）")
print("="*60)
print(f"\n最优权重:")
for name, w in zip(stock_names, w_mvp_analytic):
    print(f"  {name}: {w:.2%}")
print(f"\n组合年化收益: {mu_mvp:.2%}")
print(f"组合年化波动: {sigma_mvp:.2%}")
print(f"夏普比率:     {sharpe_mvp:.3f}")

# 手算验证
print(f"\n🔍 验证权重和为1: {w_mvp_analytic.sum():.10f}")

# 权重柱状图
fig, ax = plt.subplots(figsize=(8, 4))
colors = plt.cm.RdYlGn((w_mvp_analytic - w_mvp_analytic.min())/(w_mvp_analytic.max() - w_mvp_analytic.min() + 1e-10))
ax.bar(stock_names, w_mvp_analytic*100, color=colors, edgecolor='white')
ax.axhline(20, color='gray', linestyle='--', alpha=0.5, label='等权基准 20%')
ax.set_ylabel('权重 (%)'); ax.set_title('最小方差组合权重分配（解析解）')
ax.legend()
for i, (name, w) in enumerate(zip(stock_names, w_mvp_analytic)):
    ax.text(i, w*100 + 0.5, f'{w:.1%}', ha='center', fontsize=10)
plt.tight_layout(); plt.savefig('mvp_weights.png', dpi=150, bbox_inches='tight'); plt.show()

## 4. 方法二：scipy 数值优化（最大夏普比）

最小方差组合有闭式解，但**最大夏普比组合没有解析解**，需要数值优化。

$$\max_{\mathbf{w}} \frac{\mathbf{w}^T\boldsymbol{\mu} - r_f}{\sqrt{\mathbf{w}^T\Sigma\mathbf{w}}} \quad \text{s.t.} \quad \mathbf{1}^T\mathbf{w} = 1, \; w_i \geq 0$$

> 约束 $w_i \geq 0$ 表示**不允许做空**（实际投资中常见限制）。

In [ ]:
# ========== 数值优化：最大夏普比组合 ==========
rf = 0.02  # 无风险利率 2%

def portfolio_stats(w, mu, Sigma):
    '''计算组合收益、波动、夏普比'''
    ret = w @ mu
    vol = np.sqrt(w @ Sigma @ w)
    sharpe = (ret - rf) / vol if vol > 0 else 0
    return ret, vol, sharpe

def neg_sharpe(w, mu, Sigma):
    '''负夏普比（最小化目标）'''
    _, _, s = portfolio_stats(w, mu, Sigma)
    return -s

# 约束条件
n = len(stock_names)
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})  # 权重和为1
bounds = tuple((0, 1) for _ in range(n))  # 不允许做空

# 从等权开始优化
w0 = np.ones(n) / n
result = minimize(neg_sharpe, w0, args=(mu_annual.values, Sigma),
                  method='SLSQP', bounds=bounds, constraints=constraints)

w_msr = result.x
mu_msr, sigma_msr, sharpe_msr = portfolio_stats(w_msr, mu_annual.values, Sigma)

print("="*60)
print("📊 最大夏普比组合（scipy数值优化）")
print("="*60)
print(f"收敛: {result.success}, 迭代: {result.nit} 次")
print(f"\n最优权重:")
for name, w in zip(stock_names, w_msr):
    print(f"  {name}: {w:.2%}")
print(f"\n组合年化收益: {mu_msr:.2%}")
print(f"组合年化波动: {sigma_msr:.2%}")
print(f"夏普比率:     {sharpe_msr:.3f}")

# 对比等权组合
w_eq = np.ones(n) / n
mu_eq, sigma_eq, sharpe_eq = portfolio_stats(w_eq, mu_annual.values, Sigma)
print(f"\n📊 等权组合对比:")
print(f"  年化收益: {mu_eq:.2%}, 波动: {sigma_eq:.2%}, 夏普: {sharpe_eq:.3f}")
print(f"  MSR vs 等权: 夏普提升 {(sharpe_msr/sharpe_eq-1)*100:.1f}%")

# 权重对比图
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(n)
w = 0.35
ax.bar(x - w/2, w_mvp_analytic*100, w, label='最小方差组合', color='steelblue', edgecolor='white')
ax.bar(x + w/2, w_msr*100, w, label='最大夏普比组合', color='coral', edgecolor='white')
ax.axhline(20, color='gray', linestyle='--', alpha=0.5, label='等权基准')
ax.set_xticks(x); ax.set_xticklabels(stock_names)
ax.set_ylabel('权重 (%)'); ax.set_title('MVP vs MSR 权重对比')
ax.legend()
plt.tight_layout(); plt.savefig('mvp_vs_msr_weights.png', dpi=150, bbox_inches='tight'); plt.show()

## 5. 构建有效前沿

有效前沿是所有 Pareto 最优组合的集合：在给定期望收益下风险最小。

### 方法

对一系列目标收益 $\mu_{target}$，求解：

$$\min_{\mathbf{w}} \frac{1}{2}\mathbf{w}^T\Sigma\mathbf{w} \quad \text{s.t.} \quad \mathbf{w}^T\boldsymbol{\mu} = \mu_{target},\; \mathbf{1}^T\mathbf{w} = 1,\; w_i \geq 0$$

> 这也是「**两基金分离定理**」：有效前沿上的任意组合都可以由两个有效组合（如 MVP 和 MSR）线性组合而成。

In [ ]:
# ========== 构建有效前沿 ==========
def min_variance_for_target(target_return, mu, Sigma, allow_short=False):
    '''给定目标收益，求最小方差组合'''
    n = len(mu)
    w0 = np.ones(n)/n
    constraints = [
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        {'type': 'eq', 'fun': lambda w: w @ mu - target_return}
    ]
    bounds = tuple((-1, 1) for _ in range(n)) if allow_short else tuple((0, 1) for _ in range(n))
    
    def portfolio_variance(w):
        return w @ Sigma @ w
    
    result = minimize(portfolio_variance, w0, method='SLSQP', bounds=bounds, constraints=constraints)
    return result.x if result.success else None, np.sqrt(result.fun) if result.success else None

# 生成目标收益范围
target_returns = np.linspace(mu_mvp, max(mu_annual)*0.95, 50)
frontier_vols_no_short = []
frontier_rets_no_short = []

for tr in target_returns:
    w_opt, vol_opt = min_variance_for_target(tr, mu_annual.values, Sigma, allow_short=False)
    if w_opt is not None:
        frontier_vols_no_short.append(vol_opt)
        frontier_rets_no_short.append(tr)

# 允许做空的有效前沿（对比）
target_returns2 = np.linspace(min(mu_annual)*0.5, max(mu_annual)*1.5, 60)
frontier_vols_short = []
frontier_rets_short = []
for tr in target_returns2:
    w_opt, vol_opt = min_variance_for_target(tr, mu_annual.values, Sigma, allow_short=True)
    if w_opt is not None:
        frontier_vols_short.append(vol_opt)
        frontier_rets_short.append(tr)

# ========== 可视化 ==========
fig, ax = plt.subplots(figsize=(12, 8))

# 有效前沿
ax.plot(frontier_vols_no_short, frontier_rets_no_short, 'b-', linewidth=2.5, label='有效前沿（禁止做空）')
ax.plot(frontier_vols_short, frontier_rets_short, 'b--', linewidth=1.5, alpha=0.5, label='有效前沿（允许做空）')

# 个股
for i, name in enumerate(stock_names):
    ax.scatter(sigma_annual[i], mu_annual[i], s=100, zorder=5, label=name)
    ax.annotate(name, (sigma_annual[i], mu_annual[i]), xytext=(5, 5), textcoords='offset points', fontsize=9)

# 关键组合
ax.scatter(sigma_mvp, mu_mvp, s=200, c='steelblue', marker='*', zorder=6, edgecolors='white', linewidths=1.5, label=f'MVP (夏普={sharpe_mvp:.2f})')
ax.scatter(sigma_msr, mu_msr, s=200, c='coral', marker='*', zorder=6, edgecolors='white', linewidths=1.5, label=f'MSR (夏普={sharpe_msr:.2f})')
ax.scatter(sigma_eq, mu_eq, s=150, c='gray', marker='D', zorder=5, edgecolors='white', label=f'等权 (夏普={sharpe_eq:.2f})')

# CAL线（资本市场线）
cal_x = np.linspace(0, max(frontier_vols_no_short)*1.2, 100)
cal_y = rf + sharpe_msr * cal_x
ax.plot(cal_x, cal_y, 'r-', linewidth=1.5, alpha=0.6, label=f'CAL (夏普={sharpe_msr:.3f})')
ax.scatter(0, rf, s=80, c='red', marker='s', zorder=5, label=f'无风险利率 {rf:.0%}')

ax.set_xlabel('年化波动率')
ax.set_ylabel('年化收益率')
ax.set_title('Markowitz 有效前沿')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('efficient_frontier.png', dpi=150, bbox_inches='tight'); plt.show()

print(f"\n📊 有效前沿统计:")
print(f"  最小方差组合: 收益={mu_mvp:.2%}, 波动={sigma_mvp:.2%}")
print(f"  最大夏普组合: 收益={mu_msr:.2%}, 波动={sigma_msr:.2%}")
print(f"  等权组合:     收益={mu_eq:.2%}, 波动={sigma_eq:.2%}")
print(f"  禁止做空使有效前沿变短，但更符合实际投资限制")

## 6. 方法三：PyPortfolioOpt 实战

PyPortfolioOpt 是专门用于组合优化的 Python 库。相比手写 scipy，它提供了更丰富的功能：

- `EfficientFrontier`：有效前沿计算
- `max_sharpe()`、`min_volatility()`：一行代码求最优组合
- `plotting`：内置可视化
- `BlackLittermanModel`：BL 模型支持

In [ ]:
# ========== PyPortfolioOpt ==========
try:
    from pypfopt import EfficientFrontier, risk_models, expected_returns, plotting, CLA
    
    # 计算预期收益和协方差
    mu_ppo = expected_returns.mean_historical_return(returns)
    S_ppo = risk_models.sample_cov(returns)
    
    # === 最大夏普比组合 ===
    ef = EfficientFrontier(mu_ppo, S_ppo, weight_bounds=(0, 1))
    w_msr_ppo = ef.max_sharpe(risk_free_rate=0.02)
    cleaned_msr = ef.clean_weights()
    perf_msr = ef.portfolio_performance(risk_free_rate=0.02)
    
    print("="*60)
    print("📊 PyPortfolioOpt — 最大夏普比组合")
    print("="*60)
    print(f"权重: {dict((name, f'{w:.2%}') for name, w in zip(stock_names, cleaned_msr.values()))}")
    print(f"\n收益={perf_msr[0]:.2%}, 波动={perf_msr[1]:.2%}, 夏普={perf_msr[2]:.3f}")
    
    # === 最小方差组合 ===
    ef2 = EfficientFrontier(mu_ppo, S_ppo, weight_bounds=(0, 1))
    w_mvp_ppo = ef2.min_volatility()
    cleaned_mvp = ef2.clean_weights()
    perf_mvp = ef2.portfolio_performance(risk_free_rate=0.02)
    
    print(f"\n📊 PyPortfolioOpt — 最小方差组合")
    print(f"权重: {dict((name, f'{w:.2%}') for name, w in zip(stock_names, cleaned_mvp.values()))}")
    print(f"\n收益={perf_mvp[0]:.2%}, 波动={perf_mvp[1]:.2%}, 夏普={perf_mvp[2]:.3f}")
    
    # === 对比 scipy vs PyPortfolioOpt ===
    print(f"\n📊 scipy vs PyPortfolioOpt 对比:")
    print(f"  MSR: scipy动量={mu_msr:.2%} vs PPO={perf_msr[0]:.2%}")
    print(f"        scipy波动={sigma_msr:.2%} vs PPO={perf_msr[1]:.2%}")
    print(f"  MVP: scipy动量={mu_mvp:.2%} vs PPO={perf_mvp[0]:.2%}")
    print(f"        scipy波动={sigma_mvp:.2%} vs PPO={perf_mvp[1]:.2%}")
    
    # === 有效前沿（用CLA算法） ===
    cla = CLA(mu_ppo, S_ppo, weight_bounds=(0, 1))
    ret, vol, weights = cla.efficient_frontier(points=100)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(vol, ret, 'b-', linewidth=2, label='有效前沿 (CLA算法)')
    for i, name in enumerate(stock_names):
        ax.scatter(np.sqrt(S_ppo.iloc[i,i]*252), mu_ppo.iloc[i]*252, s=80, zorder=5, label=name)
    ax.scatter(perf_msr[1], perf_msr[0], s=200, c='coral', marker='*', zorder=6, edgecolors='white', linewidths=1.5, label=f'MSR (夏普={perf_msr[2]:.2f})')
    ax.scatter(perf_mvp[1], perf_mvp[0], s=200, c='steelblue', marker='*', zorder=6, edgecolors='white', linewidths=1.5, label=f'MVP')
    ax.set_xlabel('年化波动率'); ax.set_ylabel('年化收益率')
    ax.set_title('有效前沿 (PyPortfolioOpt CLA算法)')
    ax.legend(loc='upper left', fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig('efficient_frontier_ppo.png', dpi=150, bbox_inches='tight'); plt.show()
    
    ppo_available = True
except ImportError:
    print("PyPortfolioOpt 未安装。跳过此部分，前面的 scipy 实现已涵盖核心优化逻辑。")
    print("安装方法: pip install PyPortfolioOpt")
    ppo_available = False

## 7. 约束条件的影响分析

Markowitz 模型对约束条件非常敏感。不同的约束会产生截然不同的结果。

In [ ]:
# ========== 约束条件对比 ==========
constraint_scenarios = {
    '无约束（允许做空）': (-1, 1),
    '禁止做空 (0,1)': (0, 1),
    '单只≤40%': (0, 0.4),
    '单只≥5%且≤35%': (0.05, 0.35),
}

results = []
for label, (lower, upper) in constraint_scenarios.items():
    bounds = tuple((lower, upper) for _ in range(n))
    result = minimize(neg_sharpe, w0, args=(mu_annual.values, Sigma),
                      method='SLSQP', bounds=bounds, constraints=constraints)
    if result.success:
        ret, vol, sharpe = portfolio_stats(result.x, mu_annual.values, Sigma)
        results.append({'约束': label, '收益': ret, '波动': vol, '夏普比': sharpe, '权重': result.x})
    else:
        results.append({'约束': label, '收益': np.nan, '波动': np.nan, '夏普比': np.nan, '权重': None})

print("="*70)
print("📊 不同约束条件下的最优组合")
print("="*70)
for r in results:
    print(f"\n【{r['约束']}】")
    print(f"  收益={r['收益']:.2%}, 波动={r['波动']:.2%}, 夏普={r['夏普比']:.3f}")
    if r['权重'] is not None:
        for name, w in zip(stock_names, r['权重']):
            bar = '█' * int(abs(w)*50)
            print(f"  {name:6s}: {w:+.1%} {bar}")

# 可视化
fig, ax = plt.subplots(figsize=(10, 5))
for r in results:
    if not np.isnan(r['收益']):
        ax.scatter(r['波动'], r['收益'], s=150, label=f"{r['约束']}\n(夏普={r['夏普比']:.3f})")
ax.set_xlabel('年化波动率'); ax.set_ylabel('年化收益率')
ax.set_title('不同约束条件下的最优组合位置')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('constraint_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

print("\n💡 关键洞察：约束越紧 → 可行域越小 → 最优夏普比越低（无免费午餐）")

## 8. 分散化原理可视化

### 为什么分散化能降低风险？

两个资产的组合方差：
$$\sigma_p^2 = w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2w_1w_2\rho\sigma_1\sigma_2$$

当 $\rho < 1$ 时，$\sigma_p < w_1\sigma_1 + w_2\sigma_2$ —— **风险小于加权平均**。

当 $\rho \to -1$ 时，可以构造**零风险组合**。

In [ ]:
# ========== 分散化效果演示 ==========
# 选两只相关性最低的股票
corr_matrix = returns.corr()
min_corr_pair = np.unravel_index(np.argmin(corr_matrix.values + np.eye(n)*10), corr_matrix.shape)
s1, s2 = min_corr_pair

r1 = returns.iloc[:, s1]
r2 = returns.iloc[:, s2]

# 计算不同权重组合的风险
weights_range = np.linspace(0, 1, 100)
port_vols = []
port_rets = []
for w in weights_range:
    port_ret = w * mu_annual.iloc[s1] + (1-w) * mu_annual.iloc[s2]
    port_vol = np.sqrt(w**2 * sigma_annual.iloc[s1]**2 + (1-w)**2 * sigma_annual.iloc[s2]**2 + 2*w*(1-w)*corr_matrix.iloc[s1,s2]*sigma_annual.iloc[s1]*sigma_annual.iloc[s2])
    port_rets.append(port_ret)
    port_vols.append(port_vol)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(port_vols, port_rets, 'b-', linewidth=2, label=f'{stock_names[s1]} + {stock_names[s2]}')
ax.scatter(sigma_annual.iloc[s1], mu_annual.iloc[s1], s=120, c='coral', zorder=5, label=stock_names[s1])
ax.scatter(sigma_annual.iloc[s2], mu_annual.iloc[s2], s=120, c='steelblue', zorder=5, label=stock_names[s2])

# 纯股票连线的中点（无分散化）
mid_ret = (mu_annual.iloc[s1] + mu_annual.iloc[s2]) / 2
mid_vol = (sigma_annual.iloc[s1] + sigma_annual.iloc[s2]) / 2
ax.scatter(mid_vol, mid_ret, s=100, c='gray', marker='x', linewidths=2, label=f'无分散(等权) σ={mid_vol:.1%}')

ax.set_xlabel('年化波动率'); ax.set_ylabel('年化收益率')
ax.set_title(f'分散化效果：{stock_names[s1]} vs {stock_names[s2]} (ρ={corr_matrix.iloc[s1,s2]:.2f})')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('diversification.png', dpi=150, bbox_inches='tight'); plt.show()

print(f"\n📊 分散化效果:")
print(f"  {stock_names[s1]}: 波动={sigma_annual.iloc[s1]:.1%}")
print(f"  {stock_names[s2]}: 波动={sigma_annual.iloc[s2]:.1%}")
print(f"  两者相关性 ρ = {corr_matrix.iloc[s1,s2]:.2f}")
print(f"  简单平均波动(无分散): {mid_vol:.1%}")
print(f"  实际等权组合波动(有分散): {port_vols[50]:.1%} ← 更低！")

## 9. 小结与验收

### ✅ 验收清单

- [ ] 能用解析解（拉格朗日）求出最小方差组合权重
- [ ] 能用 scipy 数值优化求最大夏普比组合
- [ ] 理解有效前沿的构建方法和两基金分离定理
- [ ] 能解释约束条件（做空限制、权重上下限）对最优组合的影响
- [ ] 能解释分散化降低风险的数学原理（相关系数的作用）
- [ ] 使用 PyPortfolioOpt 复现优化结果

### 🔑 核心要点

| 概念 | 一句话总结 |
|------|-----------|
| 有效前沿 | 给定期望收益下风险最小的组合集合 |
| MVP | 纯最小化方差，不考虑收益 |
| MSR/切线组合 | 风险调整后收益最高 |
| 两基金分离定理 | 前沿上任一组合 = MVP + MSR 的线性组合 |
| 分散化 | ρ<1 时组合风险 < 加权平均风险 |

### ⚠️ Markowitz 模型的局限

1. **输入敏感**：期望收益的微小估计误差会导致权重剧变
2. **静态模型**：假设未来分布与历史一致
3. **不考虑高阶矩**：只关注均值和方差，忽略偏度和峰度
4. **事后优化偏差**：用全样本优化 → 回测好看但实盘打脸

> 这引出了 Black-Litterman 模型、稳健优化等进阶话题。

### 📂 生成文件

- `20_Markowitz_Portfolio_Optimization.ipynb` — 本 Notebook
- `notes/quant/20-Markowitz-均值方差优化.md` — Obsidian 精华笔记
- `efficient_frontier.png` — 有效前沿图
- `correlation_heatmap.png` — 相关性热力图
- `mvp_vs_msr_weights.png` — 权重对比图
- `diversification.png` — 分散化效果图

---

> **下节预告**：序号 21 — **阶段项目：组合优化器**，整合 CAPM + 因子 + 优化技能，构建完整组合优化工具。